# Repository Git Workflow

This notebook coordinates the CodePlayground feature branch and the website `main` branch. It defaults to read-only mode. Branch changes require a clean working tree; staging, commits, and pushes require explicit opt-in and typed confirmation. Nothing is committed or pushed automatically.

## 1. Set Up Notebook Environment for Git Commands

Use `subprocess.run` with an explicit repository working directory. The helper below captures stdout/stderr and raises on failed commands so later cells cannot mistake a Git error for success.

In [4]:
import subprocess
from pathlib import Path


def run_git(repo: Path, *args: str, check: bool = True, quiet_stderr: bool = False) -> str:
    """Run one Git command in a repository and return stdout."""
    result = subprocess.run(
        ["git", *args],
        cwd=repo,
        text=True,
        capture_output=True,
        check=False,
    )
    if result.stdout.strip():
        print(result.stdout.rstrip())
    if result.stderr.strip() and not quiet_stderr:
        print(result.stderr.rstrip())
    if check and result.returncode:
        raise RuntimeError(f"git {' '.join(args)} failed with exit code {result.returncode}")
    return result.stdout.strip()


def working_tree_is_clean(repo: Path) -> bool:
    return not run_git(repo, "status", "--porcelain")


## 2. Define Repository Paths and Target Branch Rules

Set the two working-copy paths and the intended branch for each repository. Keep the file lists narrow: an empty list is a deliberate stop, not permission to stage the entire repository.

In [2]:
CODEPLAYGROUND_REPO = Path.home() / "Documents/GitHub/CodePlayground"
WEBSITE_REPO = Path.home() / "Documents/GitHub/koulakhilesh.github.io"

CODEPLAYGROUND_BRANCH = "feature/tfl-step-free-access"
WEBSITE_BRANCH = "main"

# Populate only with reviewed, task-specific paths before using the write cells.
CODEPLAYGROUND_FILES: list[str] = []
WEBSITE_FILES: list[str] = []
CODEPLAYGROUND_COMMIT_MESSAGE = "Add step-free Tube access analysis"
WEBSITE_COMMIT_MESSAGE = "Add step-free Tube access draft"

# Writes remain disabled unless the user changes this and confirms each operation.
ALLOW_GIT_WRITES = False


## 3. Inspect Current Branch and Working Tree State

Inspect both repositories before changing branches or staging files. The helper reports dirtiness explicitly; branch switching and website pulls are guarded separately and refuse to run on a dirty tree.

In [10]:
def inspect_repository(repo: Path, label: str) -> dict[str, str | bool]:
    if not repo.is_dir():
        raise FileNotFoundError(f"Repository does not exist: {repo}")
    branch = run_git(repo, "branch", "--show-current")
    status = run_git(repo, "status", "--short")
    upstream = run_git(
        repo,
        "rev-parse",
        "--abbrev-ref",
        "--symbolic-full-name",
        "@{upstream}",
        check=False,
        quiet_stderr=True,
    )
    state = {"label": label, "branch": branch, "status": status,
             "upstream": upstream, "clean": not status}
    print(f"{label}: branch={branch}; upstream={upstream or 'none'}; clean={not status}")
    return state


def require_clean(repo: Path) -> None:
    status = run_git(repo, "status", "--short")
    if status:
        raise RuntimeError(f"Refusing branch/pull operation on a dirty tree:\n{status}")


codeplayground_state = inspect_repository(CODEPLAYGROUND_REPO, "CodePlayground")
website_state = inspect_repository(WEBSITE_REPO, "Website")


feature/tfl-step-free-access
 M london_tube_crowding/README.md
?? london_tube_crowding/step_free_access.ipynb
?? london_tube_crowding/step_free_access.py
?? london_tube_crowding/tests/test_step_free_access.py
?? scripts/repo_git_workflow.ipynb
CodePlayground: branch=feature/tfl-step-free-access; upstream=none; clean=False
main
?? _drafts/
?? assets/tube/step-free-stations.html
origin/main
Website: branch=main; upstream=origin/main; clean=False


## 4. Create and Switch to a New Branch in `codeplayground`

Reuse the target branch if already checked out. If it exists locally or on `origin`, switch to it only when the working tree is clean; otherwise create it from the current branch. The cell refuses to switch away from uncommitted work.

In [7]:
def ensure_feature_branch(repo: Path, branch: str) -> str:
    current = run_git(repo, "branch", "--show-current")
    if current == branch:
        print(f"Already on {branch}; no checkout needed.")
        return current

    require_clean(repo)
    local_match = run_git(repo, "branch", "--list", branch)
    remote_ref = f"origin/{branch}"
    remote_match = run_git(repo, "branch", "--remotes", "--list", remote_ref)

    if local_match:
        run_git(repo, "switch", branch)
    elif remote_match:
        run_git(repo, "switch", "--track", "-c", branch, remote_ref)
    else:
        run_git(repo, "switch", "-c", branch)

    switched = run_git(repo, "branch", "--show-current")
    if switched != branch:
        raise RuntimeError(f"Expected branch {branch}, found {switched}")
    return switched


# This is safe to run when the requested branch is already active.
ensure_feature_branch(CODEPLAYGROUND_REPO, CODEPLAYGROUND_BRANCH)


feature/tfl-step-free-access
Already on feature/tfl-step-free-access; no checkout needed.


'feature/tfl-step-free-access'

## 5. Apply Changes, Commit, and Push in `codeplayground`

The function below stages only the explicitly listed files. With `ALLOW_GIT_WRITES = False` it prints the intended commands and stops. To actually commit or push, the user must enable writes, provide reviewed paths, and type the exact confirmation phrases at runtime.

In [11]:
def commit_and_push(repo: Path, branch: str, files: list[str], message: str) -> str | None:
    if not files:
        print("No reviewed target files listed; nothing will be staged.")
        return None
    current = run_git(repo, "branch", "--show-current")
    if current != branch:
        raise RuntimeError(f"Refusing write on {current}; expected {branch}")

    print("Target files:", *files, sep="\n  ")
    if not ALLOW_GIT_WRITES:
        print("Read-only preview. Set ALLOW_GIT_WRITES=True and rerun to enable prompts.")
        print(f"Would run: git add -- {' '.join(files)}")
        print(f"Would run: git commit -m {message!r}")
        print(f"Would run: git push -u origin {branch}")
        return None

    staged = run_git(repo, "diff", "--cached", "--name-only").splitlines()
    if staged:
        raise RuntimeError(f"Refusing to mix with existing staged files: {staged}")
    run_git(repo, "add", "--", *files)
    staged = set(run_git(repo, "diff", "--cached", "--name-only").splitlines())
    if staged != set(files):
        raise RuntimeError(f"Staged files differ from the reviewed list: {sorted(staged)}")

    if input(f"Type COMMIT {branch} to create the commit: ") != f"COMMIT {branch}":
        print("Commit cancelled; staged changes are left for review.")
        return None
    run_git(repo, "commit", "-m", message)
    commit_hash = run_git(repo, "rev-parse", "--short", "HEAD")
    print(f"Created commit {commit_hash}")

    if input(f"Type PUSH {branch} to push to origin: ") != f"PUSH {branch}":
        print("Push skipped; the commit remains local.")
        return commit_hash
    run_git(repo, "push", "-u", "origin", branch)
    return commit_hash


codeplayground_commit = commit_and_push(
    CODEPLAYGROUND_REPO,
    CODEPLAYGROUND_BRANCH,
    CODEPLAYGROUND_FILES,
    CODEPLAYGROUND_COMMIT_MESSAGE,
)


No reviewed target files listed; nothing will be staged.


## 6. Apply Changes Directly to `main` in `website`

The website write path refuses to operate on a branch other than `main`. It only pulls with `--ff-only` when the working tree is clean; when local edits already exist, it reports that fact and leaves syncing to the user. Commit and push remain behind the same explicit confirmations.

In [12]:
def prepare_website_main(repo: Path) -> None:
    current = run_git(repo, "branch", "--show-current")
    if current != WEBSITE_BRANCH:
        raise RuntimeError(f"Website must stay on main, found {current}")

    if not working_tree_is_clean(repo):
        print("Website has local changes; skipping pull to protect them.")
        return
    if not ALLOW_GIT_WRITES:
        print("Read-only preview: would run git pull --ff-only on main.")
        return
    if input("Type PULL main to fast-forward the website: ") == "PULL main":
        run_git(repo, "pull", "--ff-only", "origin", "main")
    else:
        print("Pull skipped.")


prepare_website_main(WEBSITE_REPO)
website_commit = commit_and_push(
    WEBSITE_REPO,
    WEBSITE_BRANCH,
    WEBSITE_FILES,
    WEBSITE_COMMIT_MESSAGE,
)


main
?? _drafts/
?? assets/tube/step-free-stations.html
Website has local changes; skipping pull to protect them.
No reviewed target files listed; nothing will be staged.


## 7. Verify Final Branches, Commits, and Remote Status

This final cell is read-only. It prints each repository's active branch, latest commit summary, upstream (if configured), and concise working-tree status.

In [13]:
def final_summary(repo: Path, label: str) -> None:
    print(f"\n{label}")
    run_git(repo, "branch", "--show-current")
    run_git(repo, "log", "-1", "--oneline", "--decorate")
    upstream = run_git(
        repo,
        "rev-parse",
        "--abbrev-ref",
        "--symbolic-full-name",
        "@{upstream}",
        check=False,
        quiet_stderr=True,
    )
    print(f"upstream: {upstream or 'none'}")
    run_git(repo, "status", "-sb")


final_summary(CODEPLAYGROUND_REPO, "CodePlayground")
final_summary(WEBSITE_REPO, "Website")



CodePlayground
feature/tfl-step-free-access
43380dd (HEAD -> feature/tfl-step-free-access, origin/main, origin/HEAD, main) Merge pull request #17 from koulakhilesh/feat/london-wren-churches
upstream: none
## feature/tfl-step-free-access
 M london_tube_crowding/README.md
?? london_tube_crowding/step_free_access.ipynb
?? london_tube_crowding/step_free_access.py
?? london_tube_crowding/tests/test_step_free_access.py
?? scripts/repo_git_workflow.ipynb

Website
main
d768471 (HEAD -> main, origin/main, origin/HEAD) Refactor daily puzzle implementation and styles
origin/main
upstream: origin/main
## main...origin/main
?? _drafts/
?? assets/tube/step-free-stations.html
